# Unit 5 Lecture 5: MDOF Modal Analysis
## Modal Decomposition & Forced Response

### Learning Objectives
1. Understand modal decomposition for MDOF systems
2. Apply mode superposition method
3. Analyze forced response using modal coordinates
4. Calculate Frequency Response Functions (FRF)
5. Understand modal damping and coupling

### Context
**Building on previous lectures**:
- L3: Multi-DOF systems, normal modes, eigenvalue problems
- L4: SDOF vibrations, damping, free response
- **This lecture**: Combine them - treat each mode as SDOF!

**Why modal analysis matters**:
- **Simplification**: n-DOF system -> n independent SDOF systems
- **Physical insight**: Understand how structure vibrates
- **Design**: Target specific modes for control
- **Testing**: Experimental modal analysis (EMA)

### Key Concepts

**MDOF equation of motion**:
$$\mathbf{M}\ddot{\mathbf{x}} + \mathbf{C}\dot{\mathbf{x}} + \mathbf{K}\mathbf{x} = \mathbf{F}(t)$$

**Modal transformation**:
$$\mathbf{x}(t) = \mathbf{\Phi}\mathbf{q}(t) = \sum_{i=1}^{n} \phi_i q_i(t)$$

where:
- $\mathbf{x}$ = physical coordinates (displacements)
- $\mathbf{q}$ = modal coordinates (mode amplitudes)
- $\mathbf{\Phi}$ = modal matrix (columns are mode shapes)
- $\phi_i$ = i-th mode shape
- $q_i(t)$ = i-th modal amplitude

**Decoupled modal equations** (with proportional damping):
$$\ddot{q}_i + 2\zeta_i\omega_i\dot{q}_i + \omega_i^2 q_i = \frac{\phi_i^T \mathbf{F}(t)}{m_i}$$

Each mode behaves as independent SDOF system!

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.linalg import eig
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Plot styling
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")
print("Ready for modal analysis...")


---
## Part A: Modal Decomposition - The Key Idea

### Problem: MDOF systems are coupled

**2-DOF example**:
$$m_1\ddot{x}_1 + (k_1+k_2)x_1 - k_2x_2 = F_1(t)$$
$$m_2\ddot{x}_2 - k_2x_1 + (k_2+k_3)x_2 = F_2(t)$$

Motion of $x_1$ affects $x_2$ and vice versa (coupling terms: $-k_2x_2$, $-k_2x_1$)

### Solution: Transform to modal coordinates

**Step 1: Find natural frequencies and mode shapes**

Solve eigenvalue problem:
$$\det(\mathbf{K} - \omega^2\mathbf{M}) = 0 \quad \Rightarrow \quad \omega_1, \omega_2, ..., \omega_n$$

For each $\omega_i$, solve:
$$(\mathbf{K} - \omega_i^2\mathbf{M})\phi_i = 0 \quad \Rightarrow \quad \phi_i \text{ (mode shape)}$$

**Step 2: Normalize mode shapes**

Mass normalization:
$$\phi_i^T \mathbf{M} \phi_i = 1 \quad \text{(modal mass = 1)}$$

**Step 3: Orthogonality properties**

$$\phi_i^T \mathbf{M} \phi_j = \begin{cases} 1 & i=j \\ 0 & i\neq j \end{cases}$$
$$\phi_i^T \mathbf{K} \phi_j = \begin{cases} \omega_i^2 & i=j \\ 0 & i\neq j \end{cases}$$

These are the magic properties that decouple the system!

**Step 4: Transform coordinates**

$$\mathbf{x}(t) = \mathbf{\Phi}\mathbf{q}(t) = \phi_1 q_1(t) + \phi_2 q_2(t) + ... + \phi_n q_n(t)$$

Substitute into original equation:
$$\mathbf{M}\mathbf{\Phi}\ddot{\mathbf{q}} + \mathbf{C}\mathbf{\Phi}\dot{\mathbf{q}} + \mathbf{K}\mathbf{\Phi}\mathbf{q} = \mathbf{F}$$

Multiply by $\mathbf{\Phi}^T$:
$$\mathbf{\Phi}^T\mathbf{M}\mathbf{\Phi}\ddot{\mathbf{q}} + \mathbf{\Phi}^T\mathbf{C}\mathbf{\Phi}\dot{\mathbf{q}} + \mathbf{\Phi}^T\mathbf{K}\mathbf{\Phi}\mathbf{q} = \mathbf{\Phi}^T\mathbf{F}$$

With orthogonality:
$$\mathbf{I}\ddot{\mathbf{q}} + \text{diag}(2\zeta_i\omega_i)\dot{\mathbf{q}} + \text{diag}(\omega_i^2)\mathbf{q} = \mathbf{\Phi}^T\mathbf{F}$$

**Result: n independent SDOF equations!**
$$\ddot{q}_i + 2\zeta_i\omega_i\dot{q}_i + \omega_i^2 q_i = f_i(t)$$

where $f_i(t) = \phi_i^T \mathbf{F}(t)$ = modal force

### Physical Interpretation

**General motion** = sum of modes vibrating independently:
- Mode 1 vibrates at $\omega_1$ with amplitude $q_1(t)$
- Mode 2 vibrates at $\omega_2$ with amplitude $q_2(t)$
- Total motion: $\mathbf{x}(t) = \phi_1 q_1(t) + \phi_2 q_2(t)$

Each mode acts like independent spring-mass system!

---


In [ ]:
# TODO: Create visualization# Hint: Use matplotlib to plot your results## Suggested structure:# 1. Create figure and axes# 2. Plot calculated results# 3. Add labels and formatting# 4. Display the plot# Your code here:

---
## Part B: Forced Response - Resonance in MDOF Systems

### Harmonic Forcing

**Applied force**: $\mathbf{F}(t) = \mathbf{F}_0 \sin(\Omega t)$

where $\Omega$ = forcing frequency (rad/s)

### Modal equation for mode i:

$$\ddot{q}_i + 2\zeta_i\omega_i\dot{q}_i + \omega_i^2 q_i = \frac{\phi_i^T \mathbf{F}_0}{m_i} \sin(\Omega t)$$

Define modal force amplitude: $F_{i,0} = \phi_i^T \mathbf{F}_0$

### Steady-state response (SDOF theory!):

$$q_i(t) = Q_i \sin(\Omega t - \psi_i)$$

where:
$$Q_i = \frac{F_{i,0}/\omega_i^2}{\sqrt{(1-r_i^2)^2 + (2\zeta_i r_i)^2}}$$

$$\psi_i = \arctan\left(\frac{2\zeta_i r_i}{1-r_i^2}\right)$$

$$r_i = \frac{\Omega}{\omega_i} \quad \text{(frequency ratio)}$$

### Physical response:

$$\mathbf{x}(t) = \sum_{i=1}^{n} \phi_i Q_i \sin(\Omega t - \psi_i)$$

### Resonance Behavior

**When $\Omega \approx \omega_i$** (forcing near natural frequency):
- Mode $i$ amplitude $Q_i$ becomes very large!
- Amplification factor: $Q_i \approx \frac{1}{2\zeta_i}$ at resonance
- Other modes contribute less

**Multiple resonances**:
- n-DOF system has n resonance peaks
- Each corresponds to one natural frequency

### Frequency Response Function (FRF)

**Definition**: Response amplitude vs. forcing frequency

For excitation at DOF j, response at DOF i:
$$H_{ij}(\Omega) = \frac{X_i(\Omega)}{F_j(\Omega)} = \sum_{k=1}^{n} \frac{\phi_{ik}\phi_{jk}}{\omega_k^2 - \Omega^2 + 2i\zeta_k\omega_k\Omega}$$

**FRF magnitude** shows all natural frequencies as peaks!

**Used in**:
- Modal testing (hammer test)
- Vibration isolation design
- Structural health monitoring

---


In [ ]:
# TODO: Create visualization# Hint: Use matplotlib to plot your results## Suggested structure:# 1. Create figure and axes# 2. Plot calculated results# 3. Add labels and formatting# 4. Display the plot# Your code here:

In [ ]:
# TODO: Create visualization# Hint: Use matplotlib to plot your results## Suggested structure:# 1. Create figure and axes# 2. Plot calculated results# 3. Add labels and formatting# 4. Display the plot# Your code here:

---
## Summary

### Key Concepts

**1. Modal Decomposition**
- Transform from physical coordinates x to modal coordinates q
- Decouples equations: n-DOF -> n independent SDOF
- Each mode vibrates independently at its natural frequency

**2. Mode Superposition**
- General motion = sum of modal contributions
- $\\mathbf{x}(t) = \\sum \\phi_i q_i(t)$
- Modal amplitudes evolve according to SDOF equations

**3. Forced Response**
- Apply SDOF forced response to each mode
- Resonance occurs when forcing frequency matches natural frequency
- n natural frequencies -> n resonance peaks in FRF

**4. Frequency Response Function**
- Shows response amplitude vs. frequency
- Peaks at natural frequencies (resonances)
- Used in modal testing and system identification

### Modal Analysis Procedure

**For free vibration**:
1. Form M and K matrices
2. Solve eigenvalue problem: det(K - omega^2 M) = 0
3. Get natural frequencies omega_i and mode shapes phi_i
4. Transform initial conditions to modal coordinates
5. Solve decoupled SDOF equations
6. Transform back to physical coordinates

**For forced vibration**:
1. Calculate modal forces: F_modal = Phi^T * F
2. Solve each modal equation (SDOF forced response)
3. Sum modal responses to get physical response

### Design Implications

**Avoid resonance**:
- Keep forcing frequencies away from natural frequencies
- If unavoidable, add damping
- Change mass or stiffness to shift natural frequencies

**Modal control**:
- Target specific modes for vibration suppression
- Add damping preferentially to troublesome modes
- Tune mass dampers to specific frequencies

**Testing strategy**:
- Hammer test excites all modes
- Measure FRF to identify natural frequencies
- Extract mode shapes from multiple measurements
- Compare with finite element model

### Connection to Real Systems

**Examples**:
- **Buildings**: Ground motion (earthquake) excites multiple modes
- **Bridges**: Wind or traffic can excite resonances
- **Machinery**: Rotating imbalance creates harmonic forcing
- **Aircraft**: Turbulence excites structural modes

### Limitations

**Modal analysis assumes**:
- Linear system (superposition valid)
- Proportional damping (modes decouple)
- Time-invariant properties

**When it fails**:
- Large deformations (geometric nonlinearity)
- Material nonlinearity
- Non-proportional damping (modes couple through damping)
- Time-varying systems

### What's Next?

**Unit 5 Practical**: Modal Testing Simulation
- Hammer test simulation
- FRF measurement
- Parameter extraction
- Modal assurance criterion (MAC)

**Then**: Unit 6 (Work & Energy) and Unit 7 (Numerical Methods)

---

### Practice Problems

1. **3-DOF system**: Three equal masses in series, find all three modes
2. **Modal contribution**: Force on middle mass, which modes are excited?
3. **Damping design**: Add damper to reduce resonance peak by 50%
4. **FRF analysis**: Given measured FRF, extract omega_n and zeta

---
